# Generate Paper Embeddings with SPECTER2

This notebook creates [SPECTER2](https://github.com/allenai/SPECTER2) paper embeddings from a BigQuery table and saves the result to Google Cloud Storage as Parquet.

**Author:** [Juan Pablo Bascur](https://jpbascur.com)  
**License:** MIT

**Input:** a BigQuery table with columns: `id`, `title`, `abstract`.  
**Output:** a Parquet file on GCS with columns: `work_id` (INTEGER), `embedding` (array of 768 FLOAT64 values).

## Resume support

If the session crashes mid-run, re-run all cells. The notebook scans the checkpoint folder on Drive for already-processed papers and skips them.

## Workflow

1. Edit the **CONFIG** cell below.
2. Set **Hardware accelerator** to **T4 GPU** in *Runtime > Change runtime type*.
3. Run **Cell 1** (config + install + model load).
4. Run **Cell 2** (authenticate + load table + resume detection).
5. Run **Cell 3** (generate embeddings with checkpointing).
6. Run **Cell 4** (upload checkpoint Parquet parts to GCS).

## Troubleshooting

- If Colab runs out of GPU memory, lower `BATCH_SIZE` to `64` or `32`.
- If installs fail, set `INSTALL_STABLE_VERSION = True` and re-run Cell 1.

In [ ]:
# ============================================================
# CONFIG â€” edit these before running
# ============================================================

# BigQuery table with columns: id, title, abstract
BQ_TABLE = 'YOUR_PROJECT.YOUR_DATASET.YOUR_PAPERS_TABLE'

# Project used for BigQuery billing
BILLED_PROJECT = 'YOUR_PROJECT'

# Folder on Google Drive where checkpoint batches are saved
CHECKPOINT_DIR = 'embeddings_project/checkpoints/'

# Destination Parquet dataset folder on Google Cloud Storage
GCS_OUTPUT = 'gs://YOUR_BUCKET/dutch_embeddings/'

# Save a checkpoint every this many papers
CHECKPOINT_EVERY = 512

# Lower to 64 or 32 if Colab runs out of GPU memory
BATCH_SIZE = 128

# Set to True if installs fail for unknown reasons
INSTALL_STABLE_VERSION = False

# ============================================================
# Model constants â€” do not change
# ============================================================
MODEL_NAME    = 'allenai/specter2_base'
ADAPTER_NAME  = 'allenai/specter2'
ADAPTER_ALIAS = 'proximity'
DRIVE_ROOT    = '/content/drive/MyDrive/'

# ============================================================
# Install dependencies
# ============================================================
import subprocess
pkgs = ['adapters==1.3.0', 'transformers==4.57.6', 'gcsfs', 'google-cloud-bigquery-storage', 'db-dtypes'] if INSTALL_STABLE_VERSION \
    else ['adapters', 'transformers', 'gcsfs', 'google-cloud-bigquery-storage', 'db-dtypes']
subprocess.run(['pip', 'install', '-q'] + pkgs, check=True)

# ============================================================
# Imports
# ============================================================
import glob, os, uuid
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from adapters import AutoAdapterModel
from transformers import AutoTokenizer

# ============================================================
# Device check
# ============================================================
if torch.cuda.is_available():
    device = 'cuda'
    print(f'Using GPU: {torch.cuda.get_device_name(0)}')
else:
    device = 'cpu'
    print('Using CPU â€” this will be slow for large files.')

# ============================================================
# Load model
# ============================================================
print('Loading SPECTER2 model (first run downloads from HuggingFace)...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoAdapterModel.from_pretrained(MODEL_NAME)
model.load_adapter(ADAPTER_NAME, source='hf', load_as=ADAPTER_ALIAS)
model.set_active_adapters(ADAPTER_ALIAS)
model.to(device)
model.eval()
embedding_size = model.config.hidden_size
print(f'Model ready. Embedding size: {embedding_size}')

In [ ]:
# ============================================================
# Keep Colab session alive (run this after starting Cell 3)
# ============================================================
from IPython.display import Javascript, display
display(Javascript("""
function keepAlive() {
    document.querySelector("colab-connect-button").click();
}
setInterval(keepAlive, 60000);
"""))

In [ ]:
# ============================================================
# Authenticate, mount Drive, load table, detect resume
# ============================================================
from google.colab import auth, drive
from google.cloud import bigquery

auth.authenticate_user()
drive.mount('/content/drive')

# Load BigQuery table into RAM (~2 GB â€” fits comfortably in Colab)
print(f'Loading {BQ_TABLE}...')
client   = bigquery.Client(project=BILLED_PROJECT)
df       = client.query(f'SELECT * FROM `{BQ_TABLE}`', location='EU').to_dataframe(create_bqstorage_client=True)
print(f'Loaded {len(df)} papers.')

# Validate columns
required_columns = ['id', 'title', 'abstract']
missing = [c for c in required_columns if c not in df.columns]
if missing:
    raise ValueError(f'Missing columns in BQ table: {missing}')

# ============================================================
# Resume: scan checkpoint folder for already-processed IDs
# ============================================================
checkpoint_dir = os.path.join(DRIVE_ROOT, CHECKPOINT_DIR)
os.makedirs(checkpoint_dir, exist_ok=True)

batch_files = sorted(glob.glob(os.path.join(checkpoint_dir, 'batch_*.parquet')))

if batch_files:
    done_ids = set()
    for f in batch_files:
        t = pq.read_table(f, columns=['work_id'])
        done_ids.update(t['work_id'].to_pylist())
    df_todo = df[~df['id'].isin(done_ids)].reset_index(drop=True)
    print(f'Checkpoint found: {len(done_ids)} done, {len(df_todo)} remaining.')
else:
    df_todo = df.reset_index(drop=True)
    print(f'No checkpoint. Processing all {len(df_todo)} papers.')

In [ ]:
# ============================================================
# Generate embeddings with checkpointing
# ============================================================
def save_checkpoint(ids, emb):
    """Write one checkpoint window as a Parquet file in the checkpoint folder."""
    table = pa.table({
        'work_id': pa.array(ids.tolist(), type=pa.int64()),
        'embedding': pa.array(emb.tolist(), type=pa.list_(pa.float64(), list_size=embedding_size))
    })
    path = os.path.join(checkpoint_dir, f'batch_{uuid.uuid4().hex}.parquet')
    pq.write_table(table, path)
    return path

n = len(df_todo)
if n == 0:
    print('Nothing to do - all papers are already in the checkpoint.')
else:
    excluded_count = int((df_todo['title'].isna() | df_todo['abstract'].isna()).sum())
    if excluded_count:
        print(f'{excluded_count} papers have no title or abstract - they will get NaN embeddings.')

    for window_start in range(0, n, CHECKPOINT_EVERY):
        window_end = min(window_start + CHECKPOINT_EVERY, n)
        window = df_todo.iloc[window_start:window_end].reset_index(drop=True)
        window_embeddings = np.full((len(window), embedding_size), np.nan, dtype=np.float64)

        for start in range(0, len(window), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(window))
            batch = window.iloc[start:end]

            valid_mask = batch['title'].notna() & batch['abstract'].notna()

            if valid_mask.any():
                valid_batch = batch[valid_mask]
                texts = (
                    valid_batch['title'].astype(str)
                    + ' [SEP] '
                    + valid_batch['abstract'].astype(str)
                ).tolist()

                inputs = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors='pt')
                inputs = {k: v.to(device) for k, v in inputs.items()}

                with torch.no_grad():
                    output = model(**inputs)

                valid_positions = np.where(valid_mask.values)[0]
                window_embeddings[start + valid_positions] = (
                    output.last_hidden_state[:, 0, :].detach().cpu().numpy().astype(np.float64)
                )

            print(f'Processed {window_start + end} / {n}')

        path = save_checkpoint(window['id'], window_embeddings)
        print(f'  -> Checkpoint saved: {os.path.basename(path)}')

    print('Encoding complete.')

In [ ]:
# ============================================================
# Upload checkpoint Parquet parts to GCS
# ============================================================
import gcsfs

batch_files = sorted(glob.glob(os.path.join(checkpoint_dir, 'batch_*.parquet')))
if not batch_files:
    raise FileNotFoundError('No checkpoint batches found. Run Cell 3 first.')

output_prefix = GCS_OUTPUT.rstrip('/')
if output_prefix.endswith('.parquet'):
    raise ValueError('GCS_OUTPUT should be a folder, for example gs://YOUR_BUCKET/dutch_embeddings/')

print(f'Uploading {len(batch_files)} checkpoint files to {output_prefix}/ ...')
fs = gcsfs.GCSFileSystem(project=BILLED_PROJECT)

for i, local_path in enumerate(batch_files, start=1):
    remote_path = f'{output_prefix}/{os.path.basename(local_path)}'
    with open(local_path, 'rb') as src, fs.open(remote_path, 'wb') as dst:
        while True:
            chunk = src.read(16 * 1024 * 1024)
            if not chunk:
                break
            dst.write(chunk)
    print(f'Uploaded {i} / {len(batch_files)}: {remote_path}')

print('Done.')